<a href="https://colab.research.google.com/github/psehgal2/Pandemaniac/blob/main/Strategy6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import numpy as np
import random
import networkx as nx
from sklearn.cluster import KMeans
from sklearn.cluster import SpectralClustering

In [ ]:
class SimpleStrats:
    def __init__(self, G, num_seeds, adj_list):
        self.G = G
        self.num_seeds = num_seeds
        self.dc = nx.degree_centrality(G)
        self.ec = nx.eigenvector_centrality(G)
        self.cc = nx.closeness_centrality(G)
        self.bc = nx.betweenness_centrality(G)
        self.pr = nx.pagerank(G)
        self.harmonic = nx.harmonic_centrality(G)
        self.adj_list = adj_list

    def top_nodes_dc(self):
        sorted_nodes = sorted(self.dc, key=self.dc.get, reverse=True)
        top_nodes = sorted_nodes[:self.num_seeds]
        return top_nodes

    def top_nodes_ec(self):
        sorted_nodes = sorted(self.ec, key=self.ec.get, reverse=True)
        top_nodes = sorted_nodes[:self.num_seeds]
        return top_nodes

    def top_nodes_cc(self):
        sorted_nodes = sorted(self.cc, key=self.cc.get, reverse=True)
        top_nodes = sorted_nodes[:self.num_seeds]
        return top_nodes

    def top_nodes_bc(self):
        sorted_nodes = sorted(self.bc, key=self.bc.get, reverse=True)
        top_nodes = sorted_nodes[:self.num_seeds]
        return top_nodes

    def top_nodes_pr(self):
        sorted_nodes = sorted(self.pr, key=self.pr.get, reverse=True)
        top_nodes = sorted_nodes[:self.num_seeds]
        return top_nodes

    def top_nodes_harmonic(self):
        sorted_nodes = sorted(self.harmonic, key=self.harmonic.get, reverse=True)
        top_nodes = sorted_nodes[:self.num_seeds]
        return top_nodes

    def pick_nodes(self):
        nodes_dc = self.top_nodes_dc()
        nodes_ec = self.top_nodes_ec()
        nodes_cc = self.top_nodes_cc()

        nodes = {"degree": nodes_dc, "closeness": nodes_cc}
        result_dc_cc = run(self.adj_list, nodes)

        nodes = {"degree": nodes_dc, "eigenvector": nodes_ec}
        result_dc_ec = run(self.adj_list, nodes)

        if result_dc_cc["closeness"] > result_dc_cc["degree"] and result_dc_cc["closeness"] > result_dc_ec["eigenvector"]:
          nodes = nodes_cc
        elif result_dc_ec["eigenvector"] > result_dc_ec["degree"] and result_dc_ec["eigenvector"] > result_dc_cc["closeness"]:
          nodes = nodes_ec
        else:
          nodes = nodes_dc
        return nodes

    def pick_nodes_2(self):
        nodes_dc = self.top_nodes_dc()
        nodes_ec = self.top_nodes_ec()
        nodes_cc = self.top_nodes_cc()
        nodes_bc = self.top_nodes_bc()
        nodes_pr = self.top_nodes_pr()
        nodes_harmonic = self.top_nodes_harmonic()

        result_dc_cc = run(adj_list, {"degree": nodes_dc, "closeness": nodes_cc})
        result_dc_bc = run(adj_list, {"degree": nodes_dc, "betweenness": nodes_bc})
        result_dc_ec = run(adj_list, {"degree": nodes_dc, "eigenvector": nodes_ec})
        result_dc_pagerank = run(adj_list, {"degree": nodes_dc, "pagerank": nodes_pr})
        result_dc_harmonic = run(adj_list, {"degree": nodes_dc, "harmonic": nodes_harmonic})

        if (
            result_dc_cc["closeness"] > result_dc_cc["degree"] and
            result_dc_cc["closeness"] > result_dc_ec["eigenvector"] and
            result_dc_cc["closeness"] > result_dc_bc["betweenness"] and
            result_dc_cc["closeness"] > result_dc_pagerank["pagerank"] and
            result_dc_cc["closeness"] > result_dc_harmonic["harmonic"]
        ):
            return nodes_cc
        elif (
            result_dc_ec["eigenvector"] > result_dc_ec["degree"] and
            result_dc_ec["eigenvector"] > result_dc_cc["closeness"] and
            result_dc_ec["eigenvector"] > result_dc_bc["betweenness"] and
            result_dc_ec["eigenvector"] > result_dc_pagerank["pagerank"] and
            result_dc_ec["eigenvector"] > result_dc_harmonic["harmonic"]
        ):
            return nodes_ec
        elif (
            result_dc_bc["betweenness"] > result_dc_bc["degree"] and
            result_dc_bc["betweenness"] > result_dc_cc["closeness"] and
            result_dc_bc["betweenness"] > result_dc_ec["eigenvector"] and
            result_dc_bc["betweenness"] > result_dc_pagerank["pagerank"] and
            result_dc_bc["betweenness"] > result_dc_harmonic["harmonic"]
        ):
            return nodes_bc
        elif (
            result_dc_pagerank["pagerank"] > result_dc_pagerank["degree"] and
            result_dc_pagerank["pagerank"] > result_dc_pagerank["closeness"] and
            result_dc_pagerank["pagerank"] > result_dc_pagerank["eigenvector"] and
            result_dc_pagerank["pagerank"] > result_dc_pagerank["betweenness"] and
            result_dc_pagerank["pagerank"] > result_dc_harmonic["harmonic"]
        ):
            return nodes_pr
        elif (
            result_dc_harmonic["harmonic"] > result_dc_harmonic["degree"] and
            result_dc_harmonic["harmonic"] > result_dc_harmonic["closeness"] and
            result_dc_harmonic["harmonic"] > result_dc_harmonic["eigenvector"] and
            result_dc_harmonic["harmonic"] > result_dc_harmonic["betweenness"] and
            result_dc_harmonic["harmonic"] > result_dc_pagerank["pagerank"]
        ):
            return nodes_harmonic
        else:
            return nodes_dc






In [ ]:
'''
===========
   USAGE
===========

>>> import sim
>>> sim.run([graph], [dict with keys as names and values as a list of nodes])

Returns a dictionary containing the names and the number of nodes they got.

Example:
>>> graph = {"2": ["6", "3", "7", "2"], "3": ["2", "7, "12"], ... }
>>> nodes = {"strategy1": ["1", "5"], "strategy2": ["5", "23"], ... }
>>> sim.run(graph, nodes)
>>> {"strategy1": 243, "strategy6": 121, "strategy2": 13}

Possible Errors:
- KeyError: Will occur if any seed nodes are invalid (i.e. do not exist on the
            graph).
'''

from collections import Counter, OrderedDict
from copy import deepcopy
from random import randint


def run(adj_list, node_mappings):
  """
  Function: run
  -------------
  Runs the simulation on a graph with the given node mappings.

  adj_list: A dictionary representation of the graph adjacencies.
  node_mappings: A dictionary where the key is a name and the value is a list
                 of seed nodes associated with that name.
  """
  results = run_simulation(adj_list, node_mappings)
  return results

def run_simulations_2(adj_list, node_mappings):
    """
    Function: run_simulation
    ------------------------
    Runs the simulation. Returns a tuple with the overall result and a dictionary
    with the key as the "color"/name, and the value as the number of nodes that
    "color"/name got.

    adj_list: A dictionary representation of the graph adjacencies.
    node_mappings: A dictionary where the key is a name and the value is a list
                   of seed nodes associated with that name.
    """
    # Stores a mapping of nodes to their color.
    node_color = dict([(node, None) for node in adj_list.keys()])
    init(node_mappings, node_color)
    generation = 1

    # Keep calculating the epidemic until it stops changing. Randomly choose
    # number between 100 and 200 as the stopping point if the epidemic does not
    # converge.
    prev = None
    nodes = adj_list.keys()
    max_rounds = randint(100, 200)

    # Track the number of nodes conquered by each node
    node_conquer_count = dict([(node, 0) for node in adj_list.keys()])

    while not is_stable(generation, max_rounds, prev, node_color):
        prev = deepcopy(node_color)
        for node in nodes:
            (changed, color) = update(adj_list, prev, node)
            # Store the node's new color only if it changed.
            if changed:
                node_color[node] = color
                # Increment the conquer count for the node
                node_conquer_count[node] += 1

        # NOTE: prev contains the state of the graph of the previous generation,
        # node_colors contains the state of the graph at the current generation.
        # You could check these two dicts if you want to see the intermediate steps
        # of the epidemic.
        generation += 1

    # Return both the overall result and individual node contributions
    return get_result(node_mappings.keys(), node_color), node_conquer_count


def run_simulation(adj_list, node_mappings):
  """
  Function: run_simulation
  ------------------------
  Runs the simulation. Returns a dictionary with the key as the "color"/name,
  and the value as the number of nodes that "color"/name got.

  adj_list: A dictionary representation of the graph adjacencies.
  node_mappings: A dictionary where the key is a name and the value is a list
                 of seed nodes associated with that name.
  """
  # Stores a mapping of nodes to their color.
  node_color = dict([(node, None) for node in adj_list.keys()])
  init(node_mappings, node_color)
  generation = 1

  # Keep calculating the epidemic until it stops changing. Randomly choose
  # number between 100 and 200 as the stopping point if the epidemic does not
  # converge.
  prev = None
  nodes = adj_list.keys()
  max_rounds = randint(100, 200)
  while not is_stable(generation, max_rounds, prev, node_color):
    prev = deepcopy(node_color)
    for node in nodes:
      (changed, color) = update(adj_list, prev, node)
      # Store the node's new color only if it changSed.
      if changed: node_color[node] = color
    # NOTE: prev contains the state of the graph of the previous generation,
    # node_colros contains the state of the graph at the current generation.
    # You could check these two dicts if you want to see the intermediate steps
    # of the epidemic.
    generation += 1

  return get_result(node_mappings.keys(), node_color)


def init(color_nodes, node_color):
  """
  Function: init
  --------------
  Initializes the node to color mappings.
  """
  for (color, nodes) in color_nodes.items():
    for node in nodes:
      if node_color[node] is not None:
        node_color[node] = "__CONFLICT__"
      else:
        node_color[node] = color
  for (node, color) in node_color.items():
    if color == "__CONFLICT__":
      node_color[node] = None


def update(adj_list, node_color, node):
  """
  Function: update
  ----------------
  Updates each node based on its neighbors.
  """
  neighbors = adj_list[node]
  colored_neighbors = list(filter(None, [node_color[x] for x in neighbors]))
  total_votes = len(colored_neighbors)
  team_count = Counter(colored_neighbors)
  if node_color[node] is not None:
    team_count[node_color[node]] += 1.5
    total_votes += 1.5
  most_common = team_count.most_common(1)
  if len(most_common) > 0 and \
    most_common[0][1] > total_votes / 2.0:
    return (True, most_common[0][0])

  return (False, node_color[node])


def is_stable(generation, max_rounds, prev, curr):
  """
  Function: is_stable
  -------------------
  Checks whether or not the epidemic has stabilized.
  """
  if generation <= 1 or prev is None:
    return False
  if generation == max_rounds:
    return True
  for node, color in curr.items():
    if not prev[node] == curr[node]:
      return False
  return True


def get_result(colors, node_color):
  """
  Function: get_result
  --------------------
  Get the resulting mapping of colors to the number of nodes of that color.
  """
  color_nodes = {}
  for color in colors:
    color_nodes[color] = 0
  for node, color in node_color.items():
    if color is not None:
      color_nodes[color] += 1
  return color_nodes


In [ ]:
class GraphProcessing:
    def __init__(self, filepath):
        self.filepath = filepath
        self.graph = None
        self.format = None
        self.num_seeds = None
        self.unique_id = None
        self.adjacency = None

    def get_graph(self):
        return self.graph

    def get_format(self):
        return self.format

    def get_num_seeds(self):
        return self.num_seeds

    def get_unique_id(self):
        return self.unique_id

    def get_adjacency(self):
      return self.adjacency

    def open_sampling_file(self):
        components = self.filepath.split('.')
        competition_format = components[0]
        num_seeds = int(components[1])
        unique_id = int(components[2])

        with open(file_path, "r") as file:
            file_contents = json.load(file)
        return file_contents, competition_format, num_seeds, unique_id

    def convert_to_graph(self):
        adjacency, competition_format, num_seeds, unique_id = self.open_sampling_file()
        self.adjacency = adjacency
        G = nx.Graph(adjacency)
        assert len(adjacency) == nx.number_of_nodes(G)
        self.graph = G
        self.format = competition_format
        self.num_seeds = num_seeds
        self.unique_id = unique_id

In [ ]:
# Writing to file:
def create_file(list_nums, filename):
    final_nums = []
    for i in range(len(list_nums)):
        final_nums.append(str(list_nums[i]))
    list_nums = final_nums
    list_nums = 50*list_nums
    with open(filename, "w") as wfile:
      for num in list_nums:
        wfile.write(str(num) + '\n')

In [ ]:
file_path = "J.20.35.json"
g = GraphProcessing(file_path)
g.convert_to_graph()

G = g.get_graph()
format = g.get_format()
k = g.get_num_seeds()
id = g.get_unique_id()
adj_list = g.get_adjacency()

ss = SimpleStrats(G, k, adj_list)
ss_nodes = ss.pick_nodes()
create_file(ss_nodes, "ss_submission_J_20_35")
ss_nodes_2 = ss.pick_nodes_2()
create_file(ss_nodes_2, "ss2_submission_J_20_35")
ss_nodes_str = map(str, ss_nodes)
ss_nodes_2_str = map(str, ss_nodes_2)

node_dict = {"ss2": ss_nodes_2_str, "ss": ss_nodes_str}
print("Run: ", run(adj_list, node_dict))


# ta_nodes_2 = ["23", "148", "163", "55", "145", "109", "22", "107", "139", "24"]
# ta_nodes_1 = ["23", "148", "163", "55", "139", "6", "109", "145", "2", "22"]

KeyboardInterrupt: 